# Photometric Redshifts with a Neural Network
We'll train a neural network to measure **photometric redshift** --- i.e. estimate redshift from five band ($u, g, r, i, z$ galaxy photometry). 

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
torch.set_default_dtype(torch.float64)
%matplotlib inline

## download SDSS photometry
We'll use photometry from SDSS DR18

In [ ]:
CACHE = 'data/sdss_photoz.npz'
QUERY = """SELECT s.specObjID, s.z, s.zErr, p.ra, p.dec,
  p.dered_u, p.dered_g, p.dered_r, p.dered_i, p.dered_z,
  p.modelMagErr_u, p.modelMagErr_g, p.modelMagErr_r, p.modelMagErr_i, p.modelMagErr_z
FROM SpecObj AS s JOIN PhotoObj AS p ON s.bestObjID = p.objID
WHERE s.class = 'GALAXY' AND s.zWarning = 0 AND s.survey = 'sdss' AND p.clean = 1
  AND s.fiberID % 25 = 7"""


def download(path=CACHE):
    """Query SkyServer, apply the quality cuts, and write the cache."""
    import json, requests
    url = 'https://skyserver.sdss.org/dr18/SkyServerWS/SearchTools/SqlSearch'
    rows = json.loads(requests.get(url, params={'cmd': QUERY, 'format': 'json'},
                                   timeout=900).text)[0]['Rows']
    get = lambda key: np.array([row[key] for row in rows])
    mag = np.column_stack([get(f'dered_{b}') for b in 'ugriz'])
    err = np.column_stack([get(f'modelMagErr_{b}') for b in 'ugriz'])
    z = get('z')
    ok = ((z > 0.01) & (z < 0.8) & (err[:, 0] < 1.0) & np.all(err[:, 1:] < 0.5, axis=1)
          & np.all((mag > 10) & (mag < 25), axis=1))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, mag=mag[ok].astype(np.float32), mag_err=err[ok].astype(np.float32),
             z=z[ok].astype(np.float32), z_err=get('zErr')[ok].astype(np.float32),
             ra=get('ra')[ok], dec=get('dec')[ok], specobjid=get('specObjID')[ok], query=QUERY)


if not os.path.exists(CACHE):
    download()
data = np.load(CACHE)
mag, zspec = data['mag'].astype(float), data['z'].astype(float)
print(f'{len(zspec)} galaxies, z = {zspec.min():.3f} to {zspec.max():.3f} '
      f'(median {np.median(zspec):.3f}); median spectroscopic error {np.median(data["z_err"]):.1e}')

For our input, we'll use four **colors**: $u-g,\ g-r,\ r-i,\ i-z$, which trace the shape of the spectrum, plus the $r$ magnitude. The target is $z_{spec}$.

* We'll split the data into training, validation, and test with a 80/10/10 split.
* We standardize the data to zero mean and unit variance using the training set 

In [ ]:
X = np.column_stack([mag[:, :-1] - mag[:, 1:], mag[:, 2]])      # u-g, g-r, r-i, i-z, r
names = ['u-g', 'g-r', 'r-i', 'i-z', 'r']
n = len(zspec)

perm = np.random.default_rng(12).permutation(n)
i_tr, i_va, i_te = perm[:int(0.8 * n)], perm[int(0.8 * n):int(0.9 * n)], perm[int(0.9 * n):]
print(f'train {len(i_tr)}, validation {len(i_va)}, test {len(i_te)}')

x_mu, x_sd = X[i_tr].mean(0), X[i_tr].std(0)
z_mu, z_sd = zspec[i_tr].mean(), zspec[i_tr].std()
Xs = torch.tensor((X - x_mu) / x_sd)                 # standardized inputs
ys = torch.tensor((zspec - z_mu) / z_sd)             # standardized target
to_z = lambda y: y * z_sd + z_mu                     # back to redshift

plot the data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].scatter(zspec[i_tr], X[i_tr, 1], s=1, alpha=0.3, color='k')
axes[0].set_xlabel(r'$z_{\rm spec}$'); axes[0].set_ylabel('g - r')
axes[1].scatter(zspec[i_tr], X[i_tr, 4], s=1, alpha=0.3, color='k')
axes[1].set_xlabel(r'$z_{\rm spec}$')
axes[1].set_ylabel('r')
axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

Two populations are visible. Most galaxies are the SDSS main galaxy sample, selected to $r < 17.77$ with $z \lesssim 0.25$. The redder, fainter objects are mostly
luminous red galaxies (LRGs) targeted by color out to $z \approx 0.5$. 

## autodiff with PyTorch 
PyTorch gives you two things: **tensors** (NumPy arrays that can live on a GPU) and **automatic differentiation**. Mark a tensor with `requires_grad=True`, compute anything from it, call `.backward()` on a scalar, and the gradient appears in `.grad`.

Lets check it on something you can differentiate by hand: the Gaussian negative log-likelihood of
the training redshifts,
$$
-\log L(\mu) = \sum_i \left[\frac{(z_i-\mu)^2}{2}\right] + \text{const},
\qquad
\frac{\partial L}{\partial\mu} = -\sum_i (z_i-\mu)
$$

In [ ]:
zt = torch.tensor(zspec[i_tr])
mu = torch.tensor(0.1, requires_grad=True)

nll = torch.sum((zt - mu) ** 2/2)
nll.backward()                                   

with torch.no_grad():                            # analytic gradients, for comparison
    d_mu = -torch.sum(zt - mu)
print(f'd/dmu    : autograd {mu.grad.item():+.10e}   analytic {d_mu.item():+.10e}')

## a neural network, by hand
Let's set up a neural network with two hidden layers:

$$\mathbf{h}_1 = \mathrm{ReLU}(\mathbf{x} W_1 + \mathbf{b}_1), \qquad
\mathbf{h}_2 = \mathrm{ReLU}(\mathbf{h}_1 W_2 + \mathbf{b}_2), \qquad
f(\mathbf{x}) = \mathbf{h}_2 W_3 + b_3,$$

with $\mathrm{ReLU}(a) = \max(0, a)$ applied elementwise.

**Initialization.** We'll draw each weight matrix randomly from $\mathcal{N}(0, 2/n_{in})$ and set biases to zero. 

In [ ]:
def init_params(sizes, seed):
    ''' Weights ~ N(0, 2/n_in) (He et al. 2015), biases zero.
    Returns a flat list [W1, b1, W2, b2, ...].
    '''
    g = torch.Generator().manual_seed(seed)
    params = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        W = torch.randn(n_in, n_out, generator=g) * np.sqrt(2.0 / n_in)
        params += [W.requires_grad_(), torch.zeros(n_out, requires_grad=True)]
    return params


def forward(params, x):
    ''' evaluating the neural network forward from input through the hidden layers
    '''
    h = x
    for W, b in zip(params[0:-2:2], params[1:-2:2]):      # hidden layers
        h = torch.relu(h @ W + b)
    return h @ params[-2] + params[-1]                      # linear output layer


SIZES = [5, 64, 64, 1]
params = init_params(SIZES, seed=0)
print('parameter shapes:', [tuple(p.shape) for p in params])
print(f'total parameters: {sum(p.numel() for p in params)}')

## loss function 
Assume the standardized redshift is the network's output plus Gaussian noise of fixed width $\sigma$: $y_i \sim \mathcal{N}(f(\mathbf{x}_i), \sigma^2)$. Then
$$-\log L = \sum_i \frac{(y_i - f(\mathbf{x}_i))^2}{2\sigma^2} + \text{const},$$
so minimizing the mean squared error *is* maximum likelihood. We take $\sigma = 1$ in standardized
units (it only rescales the loss).

In [ ]:
def loss_fn(pred, y):
    ''' loss function which is the negative log likelihood
    '''
    return 0.5 * torch.mean((pred.squeeze(1) - y) ** 2)       


def train_by_hand(params, epochs=30, lr=0.05, batch=256, seed=1):
    
    g = torch.Generator().manual_seed(seed)                       
    x_tr, y_tr = Xs[i_tr], ys[i_tr]
    history = []
    
    for epoch in range(epochs):
        # minibatch order
        order = torch.randperm(len(i_tr), generator=g)
        for k in range(0, len(i_tr), batch):
            j = order[k:k + batch]
            loss = loss_fn(forward(params, x_tr[j]), y_tr[j])
            loss.backward()  # gradients for this minibatch

            # update the parameters
            with torch.no_grad():
                for p in params:
                    p -= lr * p.grad # gradient-descent step
                    p.grad.zero_()   # gradients accumulate: reset
        
        with torch.no_grad():
            history.append([loss_fn(forward(params, Xs[i]), ys[i]).item() for i in (i_tr, i_va)])
    
    return np.array(history)

In [ ]:
t0 = time.time()
hist_hand = train_by_hand(params)
print(f'30 epochs in {time.time() - t0:.1f} s')

let's examine the training and validation loss as a function of epoch

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.6))
ax.plot(hist_hand[:, 0], label='training')
ax.plot(hist_hand[:, 1], label='validation')
ax.set_xlabel('epoch')
ax.set_ylabel('loss')
ax.legend(frameon=False)

let's now check how well our photo-z neural netowrk is with our test sample

In [ ]:
# run the neural network on the test sample
with torch.no_grad():
    z_mlp = to_z(forward(params, Xs[i_te]).squeeze(1)).numpy()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.6))
ax.scatter(zspec[i_te], z_mlp, s=1, alpha=0.3, color='k')
ax.plot([0, 0.8], [0, 0.8], 'C3', lw=1)
ax.set_xlabel(r'$z_{\rm spec}$')
ax.set_ylabel(r'$z_{\rm phot}$')
ax.set_xlim(0., 0.8)
ax.set_ylim(0., 0.8)
ax.legend(frameon=False)

not bad! Let's quantify how good the fit using metrics used in the literature: 

$\sigma_{\rm NMAD} = 1.4826 \times \mathrm{median}\,|\Delta z - \mathrm{median}(\Delta z)|$, where $\Delta z = (z_{\rm phot} - z_{\rm spec})/(1+z_{\rm spec})$
and 
fraction of $\Delta z$ above 0.15

In [ ]:
def photoz_metrics(zphot, ztrue):
    dz = (zphot - ztrue) / (1 + ztrue)
    nmad = 1.4826 * np.median(np.abs(dz - np.median(dz)))
    return nmad, np.mean(np.abs(dz) > 0.15)

In [ ]:
photoz_metrics(z_mlp, zspec[i_te])

Keep this as reference. 

Now let's try adding a layer or two to the neural network.

In [ ]:
SIZES = [5, 64, 64, 64, 64, 1]
params = init_params(SIZES, seed=0)
print(f'total parameters: {sum(p.numel() for p in params)}')

Train this deeper neural network for epochs ranging from 30 to 500. How does $\sigma_{\rm NMAD}$ and outlier fraction change with the number of epochs?

Implement early stopping --- stop the training if the best validation loss does not change after 10 epochs. 

In [ ]:
def train_by_hand_earlystop(params, epochs=30, lr=0.05, batch=256, seed=1):
    g = torch.Generator().manual_seed(seed)                       
    x_tr, y_tr = Xs[i_tr], ys[i_tr]
    history = []
    
    for epoch in range(epochs):
        # minibatch order
        order = torch.randperm(len(i_tr), generator=g)
        for k in range(0, len(i_tr), batch):
            j = order[k:k + batch]
            loss = loss_fn(forward(params, x_tr[j]), y_tr[j])
            loss.backward()  # gradients for this minibatch

            # update the parameters
            with torch.no_grad():
                for p in params:
                    p -= lr * p.grad # gradient-descent step
                    p.grad.zero_()   # gradients accumulate: reset

        # evaluate validation loss 
        with torch.no_grad(): # don't take gradient descent over the validation data set


        # save best validation loss and the corresponding epoch
        # if best validation loss hasn't been updated for 10 epochs stop the training! 

    return 

How many epochs do you end up training for? What happens if you apply early stopping after 20 epcohs instead? 

Try varying the learning rate `lr=0.05` and train until early stopping. What happens when you decrease it? What happens when you increase it? Why? 

## The same network with `torch.nn` and `torch.optim`

**This is the last time in this course we'll implement a neural network from scratch.**
From here on we'll use PyTorch's built-in modules. They contain no new ideas — every piece below
does exactly what you just wrote by hand:

| what you wrote | built-in PyTorch |
|---|---|
| `init_params`, `forward` | a subclass of `torch.nn.Module` made of `nn.Linear` layers |
| `loss_fn` (Gaussian NLL with $\sigma = 1$) | `nn.functional.mse_loss` |
| `p -= lr * p.grad` and `p.grad.zero_()` | `opt.step()` and `opt.zero_grad()` from `torch.optim` |
| shuffled minibatches with `torch.randperm` | `DataLoader(TensorDataset(x, y), shuffle=True)` |
| keeping the best weights | `model.state_dict()` and `model.load_state_dict()` |

In [ ]:
import copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


class PhotozMLP(nn.Module):
    """Fully connected ReLU network. sizes = [n_in, hidden_1, ..., hidden_k, n_out]."""

    def __init__(self, sizes):
        super().__init__()
        layers = []
        for n_in, n_out in zip(sizes[:-2], sizes[1:-1]):
            layers += [nn.Linear(n_in, n_out), nn.ReLU()]          # hidden layers
        layers.append(nn.Linear(sizes[-2], sizes[-1]))             # linear output layer
        self.net = nn.Sequential(*layers)
        for layer in self.net:
            if isinstance(layer, nn.Linear):                       # He initialization
                nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)                             # shape (n,), like the targets


def nll(pred, y):
    """Gaussian negative log-likelihood with sigma = 1, averaged over galaxies."""
    return 0.5 * F.mse_loss(pred, y)


def evaluate(model, idx):
    with torch.no_grad():
        return nll(model(Xs[idx]), ys[idx]).item()


model = PhotozMLP([5, 64, 64, 1])
print(model)
print(f'total parameters: {sum(p.numel() for p in model.parameters())}')

The whole recipe looks like this: a deeper network, the **Adam** optimizer, minibatches from a **`DataLoader`**, and **early stopping** (keep the weights
from the epoch with the lowest validation loss, and stop once it hasn't improved for `patience` epochs). 

In [ ]:
def fit(model, epochs=500, lr=1e-3, weight_decay=0.0, patience=10, batch=256, seed=1):
    """Adam + minibatches + early stopping. 
    
    Returns the loss history; leaves the best weights (lowest validation loss) loaded in the model.
    """
    loader = DataLoader(TensorDataset(Xs[i_tr], ys[i_tr]), batch_size=batch, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss, best_state, best_epoch, history = np.inf, None, 0, []
    for epoch in range(epochs):
        model.train()                       # no effect here; matters for dropout / batch norm
        for xb, yb in loader:
            loss = nll(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        history.append([evaluate(model, i_tr), evaluate(model, i_va)])
        if history[-1][1] < best_loss:
            best_loss, best_epoch = history[-1][1], epoch
            best_state = copy.deepcopy(model.state_dict())
        elif epoch - best_epoch >= patience:
            break
    model.load_state_dict(best_state)
    print(f'stopped after {len(history)} epochs; best validation loss {best_loss:.4f} at epoch {best_epoch + 1}')
    return np.array(history)


torch.manual_seed(0)
deep = PhotozMLP([5, 64, 64, 64, 64, 1])
t0 = time.time()
hist_deep = fit(deep)
print(f'trained in {time.time() - t0:.1f} s')

with torch.no_grad():
    z_deep = to_z(deep(Xs[i_te])).numpy()
nmad, outl = photoz_metrics(z_deep, zspec[i_te])
print(f'test set: sigma_NMAD = {nmad:.4f}, outlier fraction = {outl:.2%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(hist_deep[:, 0], label='training')
axes[0].plot(hist_deep[:, 1], label='validation')
axes[0].axvline(np.argmin(hist_deep[:, 1]), color='0.6', ls=':', label='best epoch')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(frameon=False)
axes[1].scatter(zspec[i_te], z_deep, s=1, alpha=0.3, color='k')
axes[1].plot([0, 0.8], [0, 0.8], 'C3', lw=1)
axes[1].set_xlim(0., 0.8); axes[1].set_ylim(0., 0.8)
axes[1].set_xlabel(r'$z_{\rm spec}$'); axes[1].set_ylabel(r'$z_{\rm phot}$')
plt.tight_layout(); plt.show()